<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# **Procesamiento de Lenguaje Natural**
## **Desafio, Traductor**

**Este desafío puede realizarse de manera individual o en pareja. En caso de realizarlo en pareja, solo uno de los integrantes deberá subir el desafío.**


Nombre(s):
* Eliana Harriet

### **Consigna**

* Replicar el modelo traductor desarrollado en clase (https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/blob/set_2026/Clase%206/C%C3%B3digo/Traductor.ipynb) y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
* Modificar valores de hiperparámetros (por ejemplo, el número de unidades en las capas LSTM) y analizar su impacto en el desempeño del traductor.
* Analizar el impacto del número de neuronas en las capas recurrentes, comparando el desempeño de distintas configuraciones del modelo.
* Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
* Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones del enfoque utilizado.

### **Actividades opcionales**

* Incorporar embeddings preentrenados para ambos idiomas y evaluar su efecto sobre el rendimiento del modelo.
* Experimentar con diferentes estrategias de generación de secuencias, como muestreo aleatorio (sampling) o búsqueda por haz (beam search).
* Implementar y entrenar una versión equivalente del modelo utilizando PyTorch, comparando los resultados con la implementación original.

Importamos las librerias necesarias

In [1]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout
from tensorflow.keras.models import Model
import pickle
import os

Descargamos el dataset de traducción de Español-Inglés

In [2]:
import os, urllib.request, zipfile

DATA_ZIP = "spa-eng.zip"
DATA_URL = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
DATA_DIR = "spa-eng"
SPA_PATH = os.path.join(DATA_DIR, "spa.txt")

if not os.path.exists(SPA_PATH):
    if not os.path.exists(DATA_ZIP):
        urllib.request.urlretrieve(DATA_URL, DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(".")

# Cargar todas las líneas (dataset completo EN-ES)
with open(SPA_PATH, encoding="utf-8", errors="replace") as f:
    lines = [l.rstrip("
") for l in f]
    # quitar última línea vacía si existe
    if lines and lines[-1] == "":
        lines = lines[:-1]

print(f"Total de pares EN-ES cargados: {len(lines):,}")


Definimos el número máximo de oraciones y la semilla

In [3]:
MAX_NUM_SENTENCES = 10000
SEED = 40

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

np.random.shuffle(lines)

Cargamos los datos

In [4]:
input_sentences, output_sentences, output_sentences_inputs = [], [], []

for i, line in enumerate(lines):
    if i >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    output_sentences.append(output + ' <eos>')
    output_sentences_inputs.append('<sos> ' + output)
    input_sentences.append(input_sentence)

print(f"Oraciones cargadas: {len(input_sentences)}")

Oraciones cargadas: 10000


In [5]:
print(input_sentences[0])
print(output_sentences[0])
print(output_sentences_inputs[0])

Somebody stole my car.
Alguien robó mi auto. <eos>
<sos> Alguien robó mi auto.


Aplicamos tokenización

In [6]:
MAX_VOCAB_SIZE = 8000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

#Solo se tomarán en cuenta las 8000 palabras más frecuentes y se eliminan signos de puntuación
output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)

output_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)
output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
word2idx_outputs = output_tokenizer.word_index

num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)

max_input_len = 20
max_out_len   = 22

def _length_stats(seqs):
    lengths = np.array([len(s) for s in seqs])
    return {
        'media': lengths.mean(),
        'p50': np.percentile(lengths, 50),
        'p90': np.percentile(lengths, 90),
        'p95': np.percentile(lengths, 95),
        'p99': np.percentile(lengths, 99),
        'max': lengths.max(),
    }

in_stats  = _length_stats(input_integer_seq)
out_stats = _length_stats(output_integer_seq)

print(f"Vocabulario EN: {len(word2idx_inputs)} | ES: {len(word2idx_outputs)}")
print(f"Tokens distintos totales EN: {len(set(t for s in input_integer_seq for t in s))} | ES: {len(set(t for s in output_integer_seq for t in s))}")
print(f"Oraciones cargadas: {len(input_integer_seq)}")
print()
print(f"{'percentil':>10} | {'EN (tokens)':>12} | {'ES (tokens)':>12}")
print("-"*42)
for k in ('media','p50','p90','p95','p99','max'):
    print(f"{k:>10} | {in_stats[k]:12.1f} | {out_stats[k]:12.1f}")
print()
print(f"Cobertura vs max_input_len={max_input_len} / max_out_len={max_out_len}:")
print(f"  EN truncadas por padding: {100*np.mean([l>max_input_len for l in map(len,input_integer_seq)]):.1f}%")
print(f"  ES truncadas por padding: {100*np.mean([l>max_out_len  for l in map(len,output_integer_seq)]):.1f}%")

Vocabulario EN: 4977 | ES: 7803
Tokens distintos totales EN: 4977 | ES: 7802
Oraciones cargadas: 10000

 percentil |  EN (tokens) |  ES (tokens)
------------------------------------------
     media |          6.3 |          7.1
       p50 |          6.0 |          7.0
       p90 |         10.0 |         11.0
       p95 |         11.0 |         12.0
       p99 |         14.0 |         16.0
       max |         26.0 |         32.0

Cobertura vs max_input_len=20 / max_out_len=22:
  EN truncadas por padding: 0.1%
  ES truncadas por padding: 0.1%


Aplicamos padding a las secuencias

In [7]:
encoder_input_sequences  = pad_sequences(input_integer_seq,        maxlen=max_input_len)
decoder_input_sequences  = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_integer_seq,       maxlen=max_out_len, padding='post')

In [8]:
print(encoder_input_sequences[0])
print(decoder_input_sequences[0])
print(decoder_output_sequences[0])

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 702 752
  16 104]
[3434  157  613   21  246    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
[157 613  21 246   1   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0]


Definimos función para obtener un tf.data.Dataset.

Esta función construye un tf.data.Dataset que genera batches de (encoder_in, decoder_in) a one-hot targets.

In [9]:
def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes, repeat=False):

    n = len(enc_seqs)

    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)          # shape (max_out_len, num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    if repeat:
        ds = ds.repeat()
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(min(n, batch_size * 10), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

Aplicamos la función en el dataset para obtener el train y validation set

In [10]:
BATCH_SIZE = 64
val_split  = 0.15
split_idx  = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output, repeat=True
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output, repeat=False
)

n_train = split_idx
n_val   = len(encoder_input_sequences) - split_idx
steps_per_epoch  = int(np.ceil(n_train / BATCH_SIZE))
validation_steps = int(np.ceil(n_val / BATCH_SIZE))

print(f"train: {n_train} oraciones | {steps_per_epoch} steps/epoch")
print(f"val:   {n_val} oraciones | {validation_steps} steps/epoch")

train: 8500 oraciones | 133 steps/epoch
val:   1500 oraciones | 24 steps/epoch


Definimos función para descargar los embeddings de GloVe

In [11]:
def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID  = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

if not os.path.exists(_PKL_PATH) or not _is_valid_pickle(_PKL_PATH):
    print("Descargando gloveembedding.pkl desde Google Drive...")
    if os.path.exists(_PKL_PATH):
        os.remove(_PKL_PATH)
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=False)
    except Exception:
        os.system(f"curl -L -o {_PKL_PATH} "
                  f"'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")
    if not _is_valid_pickle(_PKL_PATH):
        raise ValueError("El archivo descargado no es un pickle válido.")
    print("Descarga completada.")
else:
    print("gloveembedding.pkl ya disponible.")

gloveembedding.pkl ya disponible.


Definimos función para cargar los embeddings de GloVe

In [12]:
def load_glove_embeddings(pkl_path):
    """Carga el pickle de GloVe y devuelve (word2idx, matrix_fn)."""
    max_bytes = 2**28 - 1
    raw = bytearray()
    sz  = os.path.getsize(pkl_path)
    with open(pkl_path, 'rb') as f:
        for _ in range(0, sz, max_bytes):
            raw += f.read(max_bytes)
    embeddings = pickle.loads(raw)
    idx_array  = np.arange(embeddings.shape[0])
    word2idx   = dict(zip(embeddings['word'], idx_array))
    return embeddings, word2idx

Definimos función para obtener el embedding por palabra

In [13]:
def get_word_embedding(word, embeddings, word2idx, n_features=50):
    i = word2idx.get(word, -1)
    return embeddings[i]['embedding'] if i != -1 else np.zeros(n_features)

Definimos función para armar la matriz de embeddings

In [14]:
def build_embedding_matrix(word2idx_inputs, embeddings, word2idx_glove,
                            nb_words, embed_dim=50):
    matrix = np.zeros((nb_words, embed_dim))
    for word, i in word2idx_inputs.items():
        if i < nb_words:
            vec = get_word_embedding(word, embeddings, word2idx_glove, embed_dim)
            if vec is not None and len(vec) > 0:
                matrix[i] = vec
    return matrix

In [15]:
EMBED_DIM = 50
# +1 obligatorio: texts_to_sequences genera un indice igual a len(word_index),
# y la fila 0 queda reservada para padding. Sin el +1, cuando el vocabulario real
# es menor que MAX_VOCAB_SIZE la matriz queda short y el Gather falla en inferencia.
nb_words  = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)

glove_embeddings, glove_word2idx = load_glove_embeddings(_PKL_PATH)
embedding_matrix = build_embedding_matrix(
    word2idx_inputs, glove_embeddings, glove_word2idx, nb_words, EMBED_DIM
)

n_rows = embedding_matrix.shape[0]
null_rows = np.sum(np.sum(embedding_matrix**2, axis=1) == 0)
pct_null = 100.0 * null_rows / n_rows

max_tok = max((max(s) if len(s) else 0) for s in input_integer_seq)

print(f"Matriz de embeddings EN: {embedding_matrix.shape} (incluye fila 0 de padding)")
print(f"Indice maximo generado por el tokenizer: {max_tok} | filas validas: 0..{n_rows-1}")
print(f"Indice dentro de rango: {max_tok < n_rows}")
print(f"Filas sin vector GloVe (OOV reales): {null_rows}/{n_rows} = {pct_null:.1f}%")

Matriz de embeddings EN: (4978, 50) (incluye fila 0 de padding)
Indice maximo generado por el tokenizer: 4977 | filas validas: 0..4977
Indice dentro de rango: True
Filas sin vector GloVe (OOV reales): 212/4978 = 4.3%


### **Modelo Seq2Seq**

In [16]:
n_units = 256
DEC_EMBED_DIM = 256   # baseline = catedra (output_dim = n_units). Step 3 varia 50/128/256

### Step 3: comparación de hiperparámetros (n_units)



Definimos función para armar el encoder. Esta función devuelve (encoder_inputs, encoder_states, encoder_emb_layer, encoder_lstm_layer) para poder reusar las capas en el modelo de inferencia.

In [17]:
def build_encoder(nb_words, embed_dim, embedding_matrix, max_input_len, n_units):

    enc_inputs   = Input(shape=(max_input_len,), name='encoder_inputs')

    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=embed_dim,
        weights=[embedding_matrix],
        trainable=False,
        name='encoder_embedding'
    )
    enc_emb = Dropout(0.3, name='encoder_dropout')(enc_emb_layer(enc_inputs))

    enc_lstm_layer = LSTM(n_units, return_state=True, name='encoder_lstm')
    _, state_h, state_c = enc_lstm_layer(enc_emb)

    return enc_inputs, [state_h, state_c], enc_emb_layer, enc_lstm_layer

Definimos función para armar el decoder. Esta función devuelve (decoder_inputs, decoder_outputs, dec_emb_layer, dec_lstm_layer, dec_dense_layer)
    para reusar las capas en inferencia.

In [18]:
def build_decoder(num_words_output, max_out_len, n_units, dec_embed_dim, encoder_states):

    dec_inputs = Input(shape=(max_out_len,), name='decoder_inputs')

    dec_emb_layer = Embedding(
        input_dim=num_words_output,
        output_dim=dec_embed_dim,
        name='decoder_embedding'
    )
    dec_emb = Dropout(0.3, name='decoder_dropout')(dec_emb_layer(dec_inputs))

    dec_lstm_layer = LSTM(n_units, return_sequences=True, return_state=True,
                          name='decoder_lstm')
    dec_out, _, _  = dec_lstm_layer(dec_emb, initial_state=encoder_states)

    dec_dense_layer = Dense(num_words_output, activation='softmax', name='decoder_dense')
    dec_out         = dec_dense_layer(dec_out)

    return dec_inputs, dec_out, dec_emb_layer, dec_lstm_layer, dec_dense_layer

Construimos encoder y decoder

In [19]:
enc_inputs, enc_states, enc_emb_layer, enc_lstm_layer = build_encoder(
    nb_words, EMBED_DIM, embedding_matrix, max_input_len, n_units
)
dec_inputs, dec_outputs, dec_emb_layer, dec_lstm_layer, dec_dense_layer = build_decoder(
    num_words_output, max_out_len, n_units, DEC_EMBED_DIM, enc_states
)

model = Model([enc_inputs, dec_inputs], dec_outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 22)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 20, 50)    │    248,900 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 22, 256)   │  1,997,824 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_dropout     │ (None, 20, 50)    │          0 │ encoder_embeddin… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dropout     │ (None, 22, 256)   │          0 │ decoder_embeddin… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    314,368 │ encoder_dropout[… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 22, 256), │    525,312 │ decoder_dropout[… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 22, 7804)  │  2,005,628 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,092,032 (19.42 MB)

 Trainable params: 4,843,132 (18.48 MB)

 Non-trainable params: 248,900 (972.27 KB)

In [20]:
model = Model([enc_inputs, dec_inputs], dec_outputs)

Definimos callbacks

In [21]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1
)
early_stop = EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
)

Compilamos el modelo

In [22]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    metrics=['accuracy']
)

Entrenamos el modelo

In [23]:
hist = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[lr_scheduler, early_stop],
    verbose=2
)

Epoch 1/30


/Users/elianah/fiuba/CEIA/CEIA-procesamiento-del-lenguaje-natural-i/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
E0000 00:00:1791257344.227098 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. U

E0000 00:00:1791257356.059929 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_4}}


133/133 - 13s - 97ms/step - accuracy: 0.6833 - loss: 3.0400 - val_accuracy: 0.7221 - val_loss: 2.0057 - learning_rate: 5.0000e-04


Epoch 2/30


...

Epoch 29/30


133/133 - 12s - 87ms/step - accuracy: 0.8166 - loss: 0.8863 - val_accuracy: 0.7847 - val_loss: 1.4478 - learning_rate: 5.0000e-04


Epoch 30/30


133/133 - 12s - 87ms/step - accuracy: 0.8198 - loss: 0.8605 - val_accuracy: 0.7840 - val_loss: 1.4463 - learning_rate: 5.0000e-04


Restoring model weights from the end of the best epoch: 30.


Visualizamos el accuracy durante el entrenamiento

In [24]:
import matplotlib.pyplot as plt
import seaborn as sns

epoch_count = range(1, len(hist.history['accuracy']) + 1)
sns.lineplot(x=epoch_count, y=hist.history['accuracy'],     label='train')
sns.lineplot(x=epoch_count, y=hist.history['val_accuracy'], label='valid')
plt.title("Accuracy durante entrenamiento")
plt.show()

### **Inferencia Modelo Seq2Seq**

Definimos el modelo Seq2Seq para realizar inferencia

In [25]:
def build_encoder_inference(enc_inputs, enc_emb_layer, enc_lstm_layer, n_units):
    """Modelo de encoder que solo devuelve los estados."""
    enc_emb = enc_emb_layer(enc_inputs)                    # reutiliza pesos entrenados
    _, state_h, state_c = enc_lstm_layer(enc_emb)
    return Model(enc_inputs, [state_h, state_c])


def build_decoder_inference(dec_emb_layer, dec_lstm_layer, dec_dense_layer, n_units):
    """Modelo de decoder paso a paso (un token por vez)."""
    dec_input_single = Input(shape=(1,), name='dec_input_single')
    dec_state_h_in   = Input(shape=(n_units,), name='dec_state_h')
    dec_state_c_in   = Input(shape=(n_units,), name='dec_state_c')

    dec_emb_single = dec_emb_layer(dec_input_single)      # reutiliza pesos entrenados
    dec_out, h_out, c_out = dec_lstm_layer(
        dec_emb_single,
        initial_state=[dec_state_h_in, dec_state_c_in]
    )
    dec_out = dec_dense_layer(dec_out)

    return Model(
        [dec_input_single, dec_state_h_in, dec_state_c_in],
        [dec_out, h_out, c_out]
    )

In [26]:
encoder_model = build_encoder_inference(enc_inputs, enc_emb_layer, enc_lstm_layer, n_units)
decoder_model = build_decoder_inference(dec_emb_layer, dec_lstm_layer, dec_dense_layer, n_units)

Aplicamos inferencia greedy

In [27]:
idx2word_input  = {v: k for k, v in word2idx_inputs.items()}
idx2word_target = {v: k for k, v in word2idx_outputs.items()}

Definimos funciones para realizar traducciones

In [28]:
def translate_sentence(input_seq):
    h, c = encoder_model.predict(input_seq, verbose=0)    # desempaqueta directo
    target_seq      = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']
    eos = word2idx_outputs['<eos>']

    output_sentence = []
    for _ in range(max_out_len):
        output_tokens, h, c = decoder_model.predict(
            [target_seq, h, c], verbose=0                 # estados como args separados
        )
        idx = np.argmax(output_tokens[0, 0, :])
        if idx == eos:
            break
        if idx > 0:
            output_sentence.append(idx2word_target[idx])
        target_seq[0, 0] = idx

    return ' '.join(output_sentence)


def translate(text):
    seq = input_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_input_len)
    return translate_sentence(seq)

Realizamos pruebas

In [29]:
print("\n--- Ejemplos del dataset ---")
for _ in range(5):
    i    = np.random.randint(len(input_sentences))
    seq  = encoder_input_sequences[i:i+1]
    pred = translate_sentence(seq)
    print(f"EN: {input_sentences[i]}")
    print(f"ES (real): {output_sentences[i].replace(' <eos>', '')}")
    print(f"ES (pred): {pred}\n")

print("--- Frases nuevas ---")
for s in ["My mother says hi.", "Where is the train station?",
          "I love learning languages.", "Can you help me?"]:
    print(f"EN: {s}")
    print(f"ES: {translate(s)}\n")


--- Ejemplos del dataset ---


E0000 00:00:1791257696.554767 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


EN: Would you prefer to do that today?
ES (real): ¿Preferirías hacerlo hoy?
ES (pred): quieres que hacer hacer

EN: He is terrible at math.
ES (real): Él es pésimo en matemáticas.
ES (pred): él es el trabajo que él



EN: Tom has left, but Mary is still here.
ES (real): Tom se ha ido, pero Mary está todavía aquí.
ES (pred): tom ha estado en el que se ha ido a mary

EN: Tom listens to jazz.
ES (real): Tomás escucha jazz.
ES (pred): tom quiere a mary



EN: I haven't ever seen one of these before.
ES (real): Nunca he visto uno de éstos hasta ahora.
ES (pred): no tengo nada de que tengo el otro pasado

--- Frases nuevas ---
EN: My mother says hi.
ES: mi hermano me dijo a tom

EN: Where is the train station?


ES: dónde está en el tren

EN: I love learning languages.
ES: sé que los niños

EN: Can you help me?


ES: me favor te puedes ayudar



## Step 2 — Extension de escala

Replicamos el modelo de la clase (10k oraciones, maxlen 20/22, vocab 8000) y ahora
**ampliamos dataset y longitud de secuencia**, que es el punto 1 de la consigna.

Decisiones tomadas a partir de la distribucion de longitudes medida en el Step 1:

| | baseline (clase) | step 2 |
|---|---|---|
| `MAX_NUM_SENTENCES` | 10 000 | **40 000** |
| `max_input_len` | 20 | **30** |
| `max_out_len` | 22 | **32** |
| `MAX_VOCAB_SIZE` | 8 000 | **15 000** |

Sobre las longitudes: el p95 del dataset es 11 tokens tanto con 10k como con 40k
oraciones, asi que subir a "p95" habria **acortado** las secuencias respecto del
baseline. Para|longitud mayor`` se sube por encima de 20/22 hasta cubrir la cola del
dataset ampliado. Para el vocabulario: con 8000 el OOV real del dataset ampliado es
0.44% en EN y **2.99%** en ES; con 15000 cae a 0.00% y 0.40%. Subirlo a 20000 solo
baja el OOV ES de 0.40% a 0.00% y encarece la capa Dense de salida.

In [30]:
# Guardamos los resultados del baseline antes de sobreescribir las variables
baseline_results = {
    'MAX_NUM_SENTENCES': 10000,
    'max_input_len': max_input_len,
    'max_out_len': max_out_len,
    'MAX_VOCAB_SIZE': MAX_VOCAB_SIZE,
    'val_loss': float(min(hist.history['val_loss'])),
    'val_accuracy': float(max(hist.history['val_accuracy'])),
    'epochs_run': len(hist.history['loss']),
    'params': int(model.count_params()),
    'steps_per_epoch': steps_per_epoch,
}
print("Baseline (step 1):", baseline_results)

Baseline (step 1): {'MAX_NUM_SENTENCES': 10000, 'max_input_len': 20, 'max_out_len': 22, 'MAX_VOCAB_SIZE': 8000, 'val_loss': 1.4463341236114502, 'val_accuracy': 0.7847272157669067, 'epochs_run': 30, 'params': 5092032, 'steps_per_epoch': 133}


### Datos: 40 000 oraciones y vocabularios mas grandes

In [31]:
MAX_NUM_SENTENCES = 40000
MAX_VOCAB_SIZE = 15000
max_input_len = 30
max_out_len   = 32

np.random.seed(SEED)
np.random.shuffle(lines)

input_sentences, output_sentences, output_sentences_inputs = [], [], []

for line in lines:
    if len(input_sentences) >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    output_sentences.append(output + ' <eos>')
    output_sentences_inputs.append('<sos> ' + output)
    input_sentences.append(input_sentence)

print(f"Oraciones cargadas: {len(input_sentences)} (baseline: {baseline_results['MAX_NUM_SENTENCES']})")

Oraciones cargadas: 40000 (baseline: 10000)


In [32]:
input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)

output_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)
output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
word2idx_outputs = output_tokenizer.word_index

num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)

print(f"Vocabulario EN: {len(word2idx_inputs)} | ES: {len(word2idx_outputs)}")
print(f"num_words_output: {num_words_output} | filas encoder: {nb_words}")

Vocabulario EN: 9122 | ES: 16054
num_words_output: 15000 | filas encoder: 9123


In [33]:
def _length_stats(seqs):
    lengths = np.array([len(s) for s in seqs])
    return {
        'media': lengths.mean(),
        'p50': np.percentile(lengths, 50),
        'p90': np.percentile(lengths, 90),
        'p95': np.percentile(lengths, 95),
        'p99': np.percentile(lengths, 99),
        'max': lengths.max(),
    }

in_stats  = _length_stats(input_integer_seq)
out_stats = _length_stats(output_integer_seq)

print(f"{'percentil':>10} | {'EN (tokens)':>12} | {'ES (tokens)':>12} | {'EN base':>9} | {'ES base':>9}")
print("-"*62)
for k in ('media','p50','p90','p95','p99','max'):
    print(f"{k:>10} | {in_stats[k]:12.1f} | {out_stats[k]:12.1f} | {'':>9} | {'':>9}")
print()
li = np.array([len(s) for s in input_integer_seq])
lo = np.array([len(s) for s in output_integer_seq])
print(f"Truncado con los nuevos maxlen ({max_input_len}/{max_out_len}):")
print(f"  EN: {100*(li>max_input_len).mean():.2f}%  (con 20 era {100*(li>20).mean():.2f}%)")
print(f"  ES: {100*(lo>max_out_len).mean():.2f}%  (con 22 era {100*(lo>22).mean():.2f}%)")

 percentil |  EN (tokens) |  ES (tokens) |   EN base |   ES base
--------------------------------------------------------------
     media |          6.3 |          7.1 |           |          
       p50 |          6.0 |          7.0 |           |          
       p90 |         10.0 |         11.0 |           |          
       p95 |         11.0 |         12.0 |           |          
       p99 |         15.0 |         16.0 |           |          
       max |         34.0 |         37.0 |           |          

Truncado con los nuevos maxlen (30/32):
  EN: 0.02%  (con 20 era 0.11%)
  ES: 0.01%  (con 22 era 0.11%)


In [34]:
encoder_input_sequences  = pad_sequences(input_integer_seq,        maxlen=max_input_len)
decoder_input_sequences  = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_integer_seq,       maxlen=max_out_len, padding='post')

print("encoder", encoder_input_sequences.shape)
print("decoder_in ", decoder_input_sequences.shape)
print("decoder_out", decoder_output_sequences.shape)

encoder (40000, 30)
decoder_in  (40000, 32)
decoder_out (40000, 32)


In [35]:
BATCH_SIZE = 64
val_split  = 0.15
split_idx  = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output, repeat=True
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output, repeat=False
)

n_train = split_idx
n_val   = len(encoder_input_sequences) - split_idx
steps_per_epoch   = int(np.ceil(n_train / BATCH_SIZE))
validation_steps = int(np.ceil(n_val / BATCH_SIZE))

print(f"train: {n_train} oraciones | {steps_per_epoch} steps/epoch (baseline: {baseline_results['steps_per_epoch']})")
print(f"val:   {n_val} oraciones | {validation_steps} steps/epoch")

train: 34000 oraciones | 532 steps/epoch (baseline: 133)
val:   6000 oraciones | 94 steps/epoch


In [36]:
embedding_matrix = build_embedding_matrix(
    word2idx_inputs, glove_embeddings, glove_word2idx, nb_words, EMBED_DIM
)

n_rows = embedding_matrix.shape[0]
null_rows = np.sum(np.sum(embedding_matrix**2, axis=1) == 0)
max_tok = max((max(s) if len(s) else 0) for s in input_integer_seq)

print(f"Matriz de embeddings EN: {embedding_matrix.shape}")
print(f"Indice maximo del tokenizer: {max_tok} | filas validas 0..{n_rows-1} | en rango: {max_tok < n_rows}")
print(f"Filas sin vector GloVe (OOV reales): {null_rows}/{n_rows} = {100.0*null_rows/n_rows:.1f}%")

Matriz de embeddings EN: (9123, 50)
Indice maximo del tokenizer: 9122 | filas validas 0..9122 | en rango: True
Filas sin vector GloVe (OOV reales): 434/9123 = 4.8%


In [37]:
enc_inputs, enc_states, enc_emb_layer, enc_lstm_layer = build_encoder(
    nb_words, EMBED_DIM, embedding_matrix, max_input_len, n_units
)
dec_inputs, dec_outputs, dec_emb_layer, dec_lstm_layer, dec_dense_layer = build_decoder(
    num_words_output, max_out_len, n_units, DEC_EMBED_DIM, enc_states
)

model = Model([enc_inputs, dec_inputs], dec_outputs)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 30, 50)    │    456,150 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 32, 256)   │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_dropout     │ (None, 30, 50)    │          0 │ encoder_embeddin… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dropout     │ (None, 32, 256)   │          0 │ decoder_embeddin… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    314,368 │ encoder_dropout[… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 32, 256), │    525,312 │ decoder_dropout[… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 32, 15000) │  3,855,000 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,990,830 (34.30 MB)

 Trainable params: 8,534,680 (32.56 MB)

 Non-trainable params: 456,150 (1.74 MB)

In [38]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    metrics=['accuracy']
)

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                                 min_lr=1e-5, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=4,
                           restore_best_weights=True, verbose=1)
print("callbacks re-creados para el modelo extendido")

callbacks re-creados para el modelo extendido


In [39]:
hist_ext = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[lr_scheduler, early_stop],
    verbose=2
)

Epoch 1/30


/Users/elianah/fiuba/CEIA/CEIA-procesamiento-del-lenguaje-natural-i/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


E0000 00:00:1791257795.980668 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_4}}


532/532 - 105s - 198ms/step - accuracy: 0.8050 - loss: 1.6865 - val_accuracy: 0.8204 - val_loss: 1.2407 - learning_rate: 5.0000e-04


Epoch 2/30


...

532/532 - 27508s - 52s/step - accuracy: 0.9260 - loss: 0.3223 - val_accuracy: 0.8853 - val_loss: 0.6636 - learning_rate: 2.5000e-04


Epoch 30/30



Epoch 30: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.


532/532 - 103s - 194ms/step - accuracy: 0.9271 - loss: 0.3172 - val_accuracy: 0.8855 - val_loss: 0.6643 - learning_rate: 2.5000e-04


Restoring model weights from the end of the best epoch: 28.


### Metrica de traduccion: BLEU

`val_accuracy` mide accuracy **por token** con teacher forcing: no dice nada sobre la
traduccion que produce el modelo en inferencia. Para eso usamos BLEU.

Dos cuidados con la API de NLTK, verificados en este entorno:

1. `sentence_bleu(references, hypothesis)` espera `references` como **lista de listas
   de tokens**. `corpus_bleu` ademas espera un elemento por hipotesis, asi que el
   anidamiento correcto es `[[ref1], [ref2], ...]`.
2. Sin `smoothing_function`, cualquier hipotesis a la que le falte un 4-gram devuelve
   **exactamente 0**. Con `method1` no.

Usamos `corpus_bleu` (agregado) y no el promedio de `sentence_bleu`: promediar
scores por oracion no es BLEU.

In [40]:
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

_smoother = SmoothingFunction().method1
BLEU_N_SAMPLES = 500


def decode_target(idx_seq):
    words = []
    for t in idx_seq:
        if t == 0:
            break
        w = idx2word_target.get(t)
        if w is None or w == '<eos>':
            break
        words.append(w)
    return words


def evaluate_bleu(encoder_model, decoder_model, n_samples=BLEU_N_SAMPLES, seed=SEED):
    rng = np.random.RandomState(seed)
    idx = rng.choice(n_val, size=min(n_samples, n_val), replace=False)

    refs, hyps = [], []
    for i in idx:
        enc_seq = encoder_input_sequences[split_idx + int(i)]
        # el encoder espera un eje de batch: pad_sequences da (30,), predict necesita (1, 30)
        hyp = translate_sentence(enc_seq[np.newaxis, :]).split()
        ref = decode_target(output_integer_seq[split_idx + int(i)])
        if len(ref) == 0 or len(hyp) == 0:
            continue
        # corpus_bleu(list_of_references, hypotheses):
        #   list_of_references -> un elemento por hipotesis, cada uno la LISTA de referencias
        #   hypotheses        -> un elemento por hipotesis, cada uno la lista de TOKENS
        refs.append([ref])
        hyps.append(hyp)

    score = corpus_bleu(refs, hyps, smoothing_function=_smoother)
    return float(score), len(refs), sum(len(h) for h in hyps), sum(len(r[0]) for r in refs)


print("helper BLEU listo")
_ref = ["el","perro","mordio","al","hombre"]
print("control perfecto  (esperado 1.0)  :",
      round(corpus_bleu([[_ref]], [_ref], smoothing_function=_smoother), 4))
print("control invertido (esperado bajo) :",
      round(corpus_bleu([[_ref]], [["hombre","al","mordio","perro","el"]],
                        smoothing_function=_smoother), 4))
print("control con 1 error (degrada, no colapsa):",
      round(corpus_bleu([[_ref]], [["el","perro","mordio","a","mujer"]],
                        smoothing_function=_smoother), 4))

helper BLEU listo
control perfecto  (esperado 1.0)  : 1.0
control invertido (esperado bajo) : 0.0803
control con 1 error (degrada, no colapsa): 0.2659


In [41]:
# Los diccionarios inversos hay que rehacerlos: el vocabulario de salida cambio
# al pasar de 8000 a 15000 tipos. Si no, se desindexarian tokens del modelo viejo.
idx2word_input  = {v: k for k, v in word2idx_inputs.items()}
idx2word_target = {v: k for k, v in word2idx_outputs.items()}

print(f"idx2word_target: {len(idx2word_target)} entradas (incluye <sos>/<eos>)")

idx2word_target: 16054 entradas (incluye <sos>/<eos>)


### Reconstruimos los modelos de inferencia sobre el modelo extendido

In [42]:
encoder_model = build_encoder_inference(enc_inputs, enc_emb_layer, enc_lstm_layer, n_units)
decoder_model = build_decoder_inference(dec_emb_layer, dec_lstm_layer, dec_dense_layer, n_units)

model.save('seq2seq_extended.keras')
encoder_model.save('seq2seq_extended_encoder.keras')
decoder_model.save('seq2seq_extended_decoder.keras')

print("modelos de inferencia reconstruidos y pesos guardados")

modelos de inferencia reconstruidos y pesos guardados


In [43]:
bleu_step2, n_eval, hyp_tokens, ref_tokens = evaluate_bleu(encoder_model, decoder_model)

extended_results = {
    'MAX_NUM_SENTENCES': MAX_NUM_SENTENCES,
    'max_input_len': max_input_len,
    'max_out_len': max_out_len,
    'MAX_VOCAB_SIZE': MAX_VOCAB_SIZE,
    'val_loss': float(min(hist_ext.history['val_loss'])),
    'val_accuracy': float(max(hist_ext.history['val_accuracy'])),
    'epochs_run': len(hist_ext.history['loss']),
    'params': int(model.count_params()),
    'steps_per_epoch': steps_per_epoch,
    'bleu': bleu_step2,
    'bleu_n': n_eval,
}

print(f"BLEU (corpus, n={n_eval}, method1): {bleu_step2:.4f}")
print(f"tokens hipotesis: {hyp_tokens} | tokens referencia: {ref_tokens}")
print()
print("COMPARACION baseline vs extendido")
print(f"{'metrica':>16} | {'baseline':>12} | {'extendido':>12}")
print("-"*46)
for k in ('MAX_NUM_SENTENCES','max_input_len','max_out_len','MAX_VOCAB_SIZE',
          'steps_per_epoch','params','epochs_run','val_loss','val_accuracy'):
    print(f"{k:>16} | {baseline_results[k]:>12} | {extended_results[k]:>12}")
print(f"{'bleu':>16} | {'n/a':>12} | {extended_results['bleu']:>12.4f}")

E0000 00:00:1791290170.043835 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


E0000 00:00:1791290175.064394 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290180.081025 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290185.081216 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290190.081377 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


...

E0000 00:00:1791290245.230567 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290250.246232 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290255.250488 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


E0000 00:00:1791290260.261449 7488714 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


BLEU (corpus, n=499, method1): 0.1160
tokens hipotesis: 3295 | tokens referencia: 3110

COMPARACION baseline vs extendido
         metrica |     baseline |    extendido
----------------------------------------------
MAX_NUM_SENTENCES |        10000 |        40000
   max_input_len |           20 |           30
     max_out_len |           22 |           32
  MAX_VOCAB_SIZE |         8000 |        15000
 steps_per_epoch |          133 |          532
          params |      5092032 |      8990830
      epochs_run |           30 |           30
        val_loss | 1.4463341236114502 | 0.6634677052497864
    val_accuracy | 0.7847272157669067 | 0.8854791522026062
            bleu |          n/a |       0.1160


### Traducciones del modelo extendido

In [44]:
def safe(fn, *a):
    try:
        return fn(*a)
    except Exception as e:
        return f"<ERROR: {type(e).__name__}: {e}>"


print("--- Ejemplos del dataset (validacion) ---")
for i in range(5):
    k = split_idx + i
    # encoder_input_sequences ya viene con padding -> (1, max_input_len) para predict
    pred = safe(translate_sentence, encoder_input_sequences[k][np.newaxis, :])
    print(f"EN: {input_sentences[k]}")
    print(f"ES (real): {output_sentences[k].replace(' <eos>','')}")
    print(f"ES (pred): {pred}")
    print()

print("--- Frases nuevas (no estaban en el entrenamiento) ---")
new_sentences = [
    "The weather is beautiful today.",
    "I would like a coffee, please.",
    "She opened the window.",
    "They are going to the cinema.",
    "This book is very interesting.",
]
for s in new_sentences:
    print(f"EN: {s}")
    print(f"ES: {safe(translate, s)}")
    print()

--- Ejemplos del dataset (validacion) ---
EN: We've altered course.
ES (real): Hemos cambiado el rumbo.
ES (pred): es importante de su parte



EN: Everyone likes her.
ES (real): A todos les gusta ella.
ES (pred): a ella le gusta el novio

EN: Your CV has really impressed me.
ES (real): Tu currículo realmente me ha impresionado.
ES (pred): tu idea me ha dicho muy bien



EN: Tom asked me if I would be willing to help.
ES (real): Tom me preguntó si estaría dispuesto a ayudar.
ES (pred): tom me pidió que quería que pudiera ir

EN: Tom was caught lying.
ES (real): A Tom lo cacharon mintiendo.
ES (pred): tom estaba asustado

--- Frases nuevas (no estaban en el entrenamiento) ---
EN: The weather is beautiful today.


ES: la nieve es muy bonita

EN: I would like a coffee, please.
ES: me gustaría una taza de café

EN: She opened the window.


ES: ella limpió la ventana

EN: They are going to the cinema.
ES: ellos están en el sur

EN: This book is very interesting.


ES: este libro es muy interesante

